# Lab Tùy chọn: Hồi quy Tuyến tính Nhiều biến

Trong lab này, bạn sẽ mở rộng các cấu trúc dữ liệu và các hàm đã xây dựng trước đó để hỗ trợ nhiều đặc trưng. Một số hàm được cập nhật khiến lab trông có vẻ dài, nhưng thực ra chỉ là những điều chỉnh nhỏ so với các hàm trước đó nên có thể xem lại nhanh chóng.
# Mục lục
- [&nbsp;&nbsp;1.1 Mục tiêu](#toc_15456_1.1)
- [&nbsp;&nbsp;1.2 Công cụ](#toc_15456_1.2)
- [&nbsp;&nbsp;1.3 Ký hiệu](#toc_15456_1.3)
- [2 Phát biểu bài toán](#toc_15456_2)
- [&nbsp;&nbsp;2.1 Ma trận X chứa các ví dụ của chúng ta](#toc_15456_2.1)
- [&nbsp;&nbsp;2.2 Vector tham số w, b](#toc_15456_2.2)
- [3 Dự đoán Mô hình với Nhiều biến](#toc_15456_3)
- [&nbsp;&nbsp;3.1 Dự đoán đơn lẻ theo từng phần tử](#toc_15456_3.1)
- [&nbsp;&nbsp;3.2 Dự đoán đơn lẻ, vector](#toc_15456_3.2)
- [4 Tính Cost với Nhiều biến](#toc_15456_4)
- [5 Gradient Descent với Nhiều biến](#toc_15456_5)
- [&nbsp;&nbsp;5.1 Tính Gradient với Nhiều biến](#toc_15456_5.1)
- [&nbsp;&nbsp;5.2 Gradient Descent với Nhiều biến](#toc_15456_5.2)
- [6 Chúc mừng](#toc_15456_6)

<a name="toc_15456_1.1"></a>
## 1.1 Mục tiêu
- Mở rộng các hàm mô hình hồi quy của chúng ta để hỗ trợ nhiều đặc trưng
    - Mở rộng cấu trúc dữ liệu để hỗ trợ nhiều đặc trưng
    - Viết lại các hàm dự đoán, cost và gradient để hỗ trợ nhiều đặc trưng
    - Sử dụng NumPy `np.dot` để vector hóa các triển khai nhằm tăng tốc độ và đơn giản hóa

<a name="toc_15456_1.2"></a>
## 1.2 Công cụ
Trong lab này, chúng ta sẽ sử dụng: 
- NumPy, một thư viện phổ biến cho tính toán khoa học
- Matplotlib, một thư viện phổ biến để vẽ đồ thị

In [ ]:
import copy, math
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
np.set_printoptions(precision=2)  # giảm độ chính xác hiển thị trên mảng numpy

<a name="toc_15456_1.3"></a>
## 1.3 Ký hiệu
Dưới đây là tóm tắt một số ký hiệu bạn sẽ gặp, đã được cập nhật cho nhiều đặc trưng.  

|Ký hiệu <img width=70/> <br />  Tổng quát  <img width=70/> | Mô tả<img width=350/>| Python (nếu có) |
|: ------------|: ------------------------------------------------------------||
| $a$ | vô hướng, không in đậm                                                      ||
| $\mathbf{a}$ | vector, in đậm                                                 ||
| $\mathbf{A}$ | ma trận, chữ hoa in đậm                                         ||
| **Hồi quy** |         |    |     |
|  $\mathbf{X}$ | ma trận ví dụ huấn luyện                  | `X_train` |   
|  $\mathbf{y}$  | mục tiêu của các ví dụ huấn luyện                | `y_train` 
|  $\mathbf{x}^{(i)}$, $y^{(i)}$ | Ví dụ huấn luyện thứ $i$ | `X[i]`, `y[i]`|
| m | số lượng ví dụ huấn luyện | `m`|
| n | số lượng đặc trưng trong mỗi ví dụ | `n`|
|  $\mathbf{w}$  |  tham số: trọng số (weight),                       | `w`    |
|  $b$           |  tham số: độ chệch (bias)                                           | `b`    |     
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | Kết quả đánh giá mô hình tại $\mathbf{x^{(i)}}$ được tham số hóa bởi $\mathbf{w},b$: $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)}+b$  | `f_wb` | 

<a name="toc_15456_2"></a>
# 2 Phát biểu bài toán

Bạn sẽ sử dụng ví dụ minh họa là dự đoán giá nhà. Tập dữ liệu huấn luyện chứa ba ví dụ với bốn đặc trưng (diện tích, số phòng ngủ, số tầng, và tuổi nhà) được thể hiện trong bảng dưới đây.  Lưu ý rằng, khác với các lab trước, diện tích ở đây tính bằng sqft thay vì 1000 sqft. Điều này gây ra một vấn đề, mà bạn sẽ giải quyết trong lab tiếp theo!

| Diện tích (sqft) | Số phòng ngủ  | Số tầng | Tuổi nhà | Giá (1000 đô la)  |   
| ----------------| ------------------- |----------------- |--------------|-------------- |  
| 2104            | 5                   | 1                | 45           | 460           |  
| 1416            | 3                   | 2                | 40           | 232           |  
| 852             | 2                   | 1                | 35           | 178           |  

Bạn sẽ xây dựng một mô hình hồi quy tuyến tính sử dụng các giá trị này để sau đó có thể dự đoán giá cho các căn nhà khác. Ví dụ, một căn nhà 1200 sqft, 3 phòng ngủ, 1 tầng, 40 năm tuổi.  

Hãy chạy ô mã sau để tạo các biến `X_train` và `y_train`.

In [ ]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

<a name="toc_15456_2.1"></a>
## 2.1 Ma trận X chứa các ví dụ của chúng ta
Tương tự như bảng ở trên, các ví dụ được lưu trữ trong một ma trận NumPy `X_train`. Mỗi hàng của ma trận biểu diễn một ví dụ. Khi bạn có $m$ ví dụ huấn luyện ( $m$ là ba trong ví dụ của chúng ta), và có $n$ đặc trưng (bốn trong ví dụ của chúng ta), $\mathbf{X}$ là một ma trận có kích thước ($m$, $n$) (m hàng, n cột).


$$\mathbf{X} = 
\begin{pmatrix}
 x^{(0)}_0 & x^{(0)}_1 & \cdots & x^{(0)}_{n-1} \\ 
 x^{(1)}_0 & x^{(1)}_1 & \cdots & x^{(1)}_{n-1} \\
 \cdots \\
 x^{(m-1)}_0 & x^{(m-1)}_1 & \cdots & x^{(m-1)}_{n-1} 
\end{pmatrix}
$$
ký hiệu:
- $\mathbf{x}^{(i)}$ là vector chứa ví dụ i. $\mathbf{x}^{(i)}$ $ = (x^{(i)}_0, x^{(i)}_1, \cdots,x^{(i)}_{n-1})$
- $x^{(i)}_j$ là phần tử j trong ví dụ i. Chỉ số trên trong ngoặc đơn biểu thị số thứ tự ví dụ trong khi chỉ số dưới biểu thị một phần tử.  

Hiển thị dữ liệu đầu vào.

In [ ]:
# dữ liệu được lưu trong mảng/ma trận numpy
print(f"X Shape: {X_train.shape}, X Type:{type(X_train)})")
print(X_train)
print(f"y Shape: {y_train.shape}, y Type:{type(y_train)})")
print(y_train)

<a name="toc_15456_2.2"></a>
## 2.2 Vector tham số w, b

* $\mathbf{w}$ là một vector có $n$ phần tử.
  - Mỗi phần tử chứa tham số liên kết với một đặc trưng.
  - trong tập dữ liệu của chúng ta, n bằng 4.
  - theo quy ước, chúng ta vẽ nó như một vector cột

$$\mathbf{w} = \begin{pmatrix}
w_0 \\ 
w_1 \\
\cdots\\
w_{n-1}
\end{pmatrix}
$$
* $b$ là một tham số vô hướng.

Để minh họa, $\mathbf{w}$ và $b$ sẽ được nạp với một số giá trị khởi tạo được chọn gần với giá trị tối ưu. $\mathbf{w}$ là một vector NumPy 1-D.

In [ ]:
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])
print(f"w_init shape: {w_init.shape}, b_init type: {type(b_init)}")

<a name="toc_15456_3"></a>
# 3 Dự đoán Mô hình với Nhiều biến
Dự đoán của mô hình với nhiều biến được cho bởi mô hình tuyến tính:

$$ f_{\mathbf{w},b}(\mathbf{x}) =  w_0x_0 + w_1x_1 +... + w_{n-1}x_{n-1} + b \tag{1}$$
hoặc theo ký hiệu vector:
$$ f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b  \tag{2} $$ 
trong đó $\cdot$ là `tích vô hướng` (dot product) của vector

Để minh họa tích vô hướng, chúng ta sẽ triển khai dự đoán sử dụng (1) và (2).

<a name="toc_15456_3.1"></a>
## 3.1 Dự đoán đơn lẻ theo từng phần tử
Dự đoán trước đó của chúng ta nhân một giá trị đặc trưng với một tham số và cộng thêm một tham số bias. Một cách mở rộng trực tiếp từ triển khai dự đoán trước đó cho nhiều đặc trưng sẽ là triển khai (1) ở trên bằng cách sử dụng vòng lặp qua từng phần tử, thực hiện phép nhân với tham số của nó rồi cộng tham số bias ở cuối cùng.

In [ ]:
def predict_single_loop(x, w, b): 
    """
    dự đoán đơn lẻ sử dụng hồi quy tuyến tính
    
    Args:
      x (ndarray): Kích thước (n,) ví dụ với nhiều đặc trưng
      w (ndarray): Kích thước (n,) tham số mô hình    
      b (scalar):  tham số mô hình     
      
    Returns:
      p (scalar):  dự đoán
    """
    n = x.shape[0]
    p = 0
    for i in range(n):
        p_i = x[i] * w[i]  
        p = p + p_i         
    p = p + b                
    return p

In [ ]:
# lấy một hàng từ dữ liệu huấn luyện của chúng ta
x_vec = X_train[0,:]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# đưa ra một dự đoán
f_wb = predict_single_loop(x_vec, w_init, b_init)
print(f"f_wb shape {f_wb.shape}, prediction: {f_wb}")

Lưu ý kích thước của `x_vec`. Đó là một vector NumPy 1-D với 4 phần tử, (4,). Kết quả, `f_wb` là một giá trị vô hướng.

<a name="toc_15456_3.2"></a>
## 3.2 Dự đoán đơn lẻ, vector

Lưu ý rằng phương trình (1) ở trên có thể được triển khai bằng cách sử dụng tích vô hướng như trong (2) ở trên. Chúng ta có thể sử dụng các phép toán vector để tăng tốc dự đoán.

Nhớ lại từ lab Python/Numpy rằng NumPy `np.dot()`[[link](https://numpy.org/doc/stable/reference/generated/numpy.dot.html)] có thể được sử dụng để thực hiện tích vô hướng của vector.

In [ ]:
def predict(x, w, b): 
    """
    dự đoán đơn lẻ sử dụng hồi quy tuyến tính
    Args:
      x (ndarray): Kích thước (n,) ví dụ với nhiều đặc trưng
      w (ndarray): Kích thước (n,) tham số mô hình   
      b (scalar):             tham số mô hình 
      
    Returns:
      p (scalar):  dự đoán
    """
    p = np.dot(x, w) + b     
    return p    

In [ ]:
# lấy một hàng từ dữ liệu huấn luyện của chúng ta
x_vec = X_train[0,:]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# đưa ra một dự đoán
f_wb = predict(x_vec,w_init, b_init)
print(f"f_wb shape {f_wb.shape}, prediction: {f_wb}")

Kết quả và kích thước giống với phiên bản trước sử dụng vòng lặp. Từ đây trở đi, `np.dot` sẽ được sử dụng cho các phép toán này. Dự đoán bây giờ là một câu lệnh duy nhất. Hầu hết các hàm sẽ triển khai trực tiếp thay vì gọi một hàm predict riêng biệt.

<a name="toc_15456_4"></a>
# 4 Tính Cost với Nhiều biến
Phương trình cho hàm cost với nhiều biến $J(\mathbf{w},b)$ là:
$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2 \tag{3}$$ 
trong đó:
$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b  \tag{4} $$ 


Khác với các lab trước, $\mathbf{w}$ và $\mathbf{x}^{(i)}$ là các vector thay vì các giá trị vô hướng, hỗ trợ nhiều đặc trưng.

Dưới đây là một triển khai của các phương trình (3) và (4). Lưu ý rằng phần này sử dụng một *mẫu chuẩn cho khóa học này* trong đó một vòng lặp for qua tất cả `m` ví dụ được sử dụng.

In [ ]:
def compute_cost(X, y, w, b): 
    """
    tính cost
    Args:
      X (ndarray (m,n)): Dữ liệu, m ví dụ với n đặc trưng
      y (ndarray (m,)) : giá trị mục tiêu
      w (ndarray (n,)) : tham số mô hình  
      b (scalar)       : tham số mô hình
      
    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):                                
        f_wb_i = np.dot(X[i], w) + b           #(n,)(n,) = scalar (xem np.dot)
        cost = cost + (f_wb_i - y[i])**2       #scalar
    cost = cost / (2 * m)                      #scalar    
    return cost

In [ ]:
# Tính và hiển thị cost bằng cách sử dụng các tham số tối ưu đã chọn trước. 
cost = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at optimal w : {cost}')

**Kết quả mong đợi**: Cost at optimal w : 1.5578904045996674e-12

<a name="toc_15456_5"></a>
# 5 Gradient Descent với Nhiều biến
Gradient descent cho nhiều biến:

$$\begin{align*} \text{lặp lại}&\text{ cho đến khi hội tụ:} \; \lbrace \newline\;
& w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{5}  \; & \text{for j = 0..n-1}\newline
&b\ \ = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

trong đó, n là số lượng đặc trưng, các tham số $w_j$,  $b$, được cập nhật đồng thời và trong đó  

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{6}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{7}
\end{align}
$$
* m là số lượng ví dụ huấn luyện trong tập dữ liệu

    
*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ là dự đoán của mô hình, trong khi $y^{(i)}$ là giá trị mục tiêu

<a name="toc_15456_5.1"></a>
## 5.1 Tính Gradient với Nhiều biến
Một triển khai để tính các phương trình (6) và (7) được thể hiện bên dưới. Có nhiều cách để triển khai điều này. Trong phiên bản này, có
- một vòng lặp ngoài qua tất cả m ví dụ. 
    - $\frac{\partial J(\mathbf{w},b)}{\partial b}$ cho ví dụ đó có thể được tính trực tiếp và cộng dồn
    - trong một vòng lặp thứ hai qua tất cả n đặc trưng:
        - $\frac{\partial J(\mathbf{w},b)}{\partial w_j}$ được tính cho mỗi $w_j$.

In [ ]:
def compute_gradient(X, y, w, b): 
    """
    Tính gradient cho hồi quy tuyến tính 
    Args:
      X (ndarray (m,n)): Dữ liệu, m ví dụ với n đặc trưng
      y (ndarray (m,)) : giá trị mục tiêu
      w (ndarray (n,)) : tham số mô hình  
      b (scalar)       : tham số mô hình
      
    Returns:
      dj_dw (ndarray (n,)): Gradient của cost theo các tham số w. 
      dj_db (scalar):       Gradient của cost theo tham số b. 
    """
    m,n = X.shape           #(số ví dụ, số đặc trưng)
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]   
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]    
        dj_db = dj_db + err                        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m                                
        
    return dj_db, dj_dw

In [ ]:
#Tính và hiển thị gradient 
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
print(f'dj_db at initial w,b: {tmp_dj_db}')
print(f'dj_dw at initial w,b: \n {tmp_dj_dw}')

**Kết quả mong đợi**:   
dj_db at initial w,b: -1.6739251122999121e-06  
dj_dw at initial w,b:   
 [-2.73e-03 -6.27e-06 -2.22e-06 -6.92e-05]

<a name="toc_15456_5.2"></a>
## 5.2 Gradient Descent với Nhiều biến
Hàm bên dưới triển khai phương trình (5) ở trên.

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Thực hiện batch gradient descent để học w và b. Cập nhật w và b bằng cách thực hiện 
    num_iters bước gradient với tốc độ học alpha
    
    Args:
      X (ndarray (m,n))   : Dữ liệu, m ví dụ với n đặc trưng
      y (ndarray (m,))    : giá trị mục tiêu
      w_in (ndarray (n,)) : giá trị khởi tạo của tham số mô hình  
      b_in (scalar)       : giá trị khởi tạo của tham số mô hình
      cost_function       : hàm để tính cost
      gradient_function   : hàm để tính gradient
      alpha (float)       : Tốc độ học
      num_iters (int)     : số lần lặp để chạy gradient descent
      
    Returns:
      w (ndarray (n,)) : Giá trị đã cập nhật của các tham số 
      b (scalar)       : Giá trị đã cập nhật của tham số 
      """
    
    # Một mảng để lưu cost J và các giá trị w tại mỗi lần lặp, chủ yếu để vẽ đồ thị sau này
    J_history = []
    w = copy.deepcopy(w_in)  #tránh việc thay đổi w toàn cục bên trong hàm
    b = b_in
    
    for i in range(num_iters):

        # Tính gradient và cập nhật các tham số
        dj_db,dj_dw = gradient_function(X, y, w, b)   

        # Cập nhật Tham số sử dụng w, b, alpha và gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
      
        # Lưu cost J tại mỗi lần lặp
        if i<100000:      # ngăn cạn kiệt tài nguyên 
            J_history.append( cost_function(X, y, w, b))

        # In cost mỗi 10 khoảng lần lặp, hoặc nhiều lần lặp hơn nếu < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")
        
    return w, b, J_history #trả về w,b cuối cùng và lịch sử J để vẽ đồ thị

Trong ô tiếp theo bạn sẽ kiểm tra việc triển khai.

In [ ]:
# khởi tạo tham số
initial_w = np.zeros_like(w_init)
initial_b = 0.
# một số thiết lập gradient descent
iterations = 1000
alpha = 5.0e-7
# chạy gradient descent 
w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient, 
                                                    alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
m,_ = X_train.shape
for i in range(m):
    print(f"prediction: {np.dot(X_train[i], w_final) + b_final:0.2f}, target value: {y_train[i]}")

**Kết quả mong đợi**:    
b,w found by gradient descent: -0.00,[ 0.2   0.   -0.01 -0.07]   
prediction: 426.19, target value: 460  
prediction: 286.17, target value: 232  
prediction: 171.47, target value: 178  

In [ ]:
# vẽ đồ thị cost theo số lần lặp  
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist)
ax2.plot(100 + np.arange(len(J_hist[100:])), J_hist[100:])
ax1.set_title("Cost vs. iteration");  ax2.set_title("Cost vs. iteration (tail)")
ax1.set_ylabel('Cost')             ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')   ;  ax2.set_xlabel('iteration step') 
plt.show()

*Những kết quả này không mấy ấn tượng*! Cost vẫn đang giảm và các dự đoán của chúng ta chưa thật sự chính xác. Lab tiếp theo sẽ khám phá cách cải thiện điều này.


<a name="toc_15456_6"></a>
# 6 Chúc mừng!
Trong lab này bạn đã:
- Xây dựng lại các hàm cho hồi quy tuyến tính, giờ đây với nhiều biến.
- Sử dụng NumPy `np.dot` để vector hóa các triển khai